In [7]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers.optimization import Adafactor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import os

MODEL_NAME = "NoYo25/BiodivBERT"
TRAIN_FILE = "species_525_cleaned.csv"
TEST_FILE = "Data_cleaned.csv"
OUTPUT_FILE = "biodivbert_result.csv"

MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 4


# DATA PREPARATION

def load_and_label_data(file_path, tokenizer):
    df = pd.read_csv(file_path)
    df['text'] = df['Title'].fillna('') + " " + df['Description'].fillna('')
    data = []

    print(f"Loading data from {file_path} ({len(df)} rows)...")

    for idx, row in df.iterrows():
        text = str(row['text'])
        species_raw = str(row['Species'])
        if species_raw == 'nan': continue
        species_list = [s.strip() for s in species_raw.split(',') if s.strip()]

        char_labels = np.zeros(len(text), dtype=int)
        found_any = False
        for species in species_list:
            start_idx = text.find(species)
            if start_idx != -1:
                found_any = True
                end_idx = start_idx + len(species)
                char_labels[start_idx] = 1
                char_labels[start_idx+1:end_idx] = 2

        if not found_any: continue

        tokenized = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_offsets_mapping=True)
        labels = []
        for (start, end) in tokenized['offset_mapping']:
            if start == end: labels.append(-100)
            else: labels.append(char_labels[start])

        data.append({
            'input_ids': torch.tensor(tokenized['input_ids']),
            'attention_mask': torch.tensor(tokenized['attention_mask']),
            'labels': torch.tensor(labels)
        })

    return data

class NERDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


# TRAINING

def train_experiment(unfreeze_layers, train_loader, val_loader):
    print(f"Training: Unfreezing Last {unfreeze_layers} Layers")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=3)

    for param in model.parameters(): param.requires_grad = False
    for param in model.classifier.parameters(): param.requires_grad = True
    if unfreeze_layers > 0:
        for layer in model.bert.encoder.layer[-unfreeze_layers:]:
            for param in layer.parameters(): param.requires_grad = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    optimizer = Adafactor(model.parameters(), lr=1e-3, eps=(1e-30, 1e-3), clip_threshold=1.0, decay_rate=-0.8, beta1=None, weight_decay=0.0, relative_step=False, scale_parameter=False, warmup_init=False)

    model.train()
    for epoch in range(EPOCHS):
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=mask, labels=labels)
            outputs.loss.backward()
            optimizer.step()

    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=mask, labels=labels)
            preds = torch.argmax(outputs.logits, dim=2)

            active_loss = labels.view(-1) != -100
            predictions.extend(preds.view(-1)[active_loss].cpu().numpy())
            true_labels.extend(labels.view(-1)[active_loss].cpu().numpy())

    return model, accuracy_score(true_labels, predictions)


# INFERENCE UTILS

def extract_species(text, model, tokenizer):
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad(): outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu())

    extracted, current = [], []
    for token, label in zip(tokens, predictions):
        if token in ["[CLS]", "[SEP]", "[PAD]"]: continue
        if label == 1:
            if current: extracted.append(tokenizer.convert_tokens_to_string(current))
            current = [token]
        elif label == 2:
            if current: current.append(token)
            else: current = [token]
        else:
            if current: extracted.append(tokenizer.convert_tokens_to_string(current)); current = []
    if current: extracted.append(tokenizer.convert_tokens_to_string(current))
    return list(set([s.replace(" ##", "") for s in extracted]))

def calculate_row_metrics(row):
    y_true = set(row['Ground_Truth_List'])
    y_pred = set(row['Extracted_List'])
    misspelled = list(y_pred - y_true)

    if not y_true and not y_pred: return pd.Series([misspelled, 1.0, 1.0, 1.0])
    if not y_true or not y_pred: return pd.Series([misspelled, 0.0, 0.0, 0.0])

    matches = len(y_true.intersection(y_pred))
    p = matches / len(y_pred)
    r = matches / len(y_true)
    f1 = 2 * (p*r) / (p+r) if (p+r) > 0 else 0
    return pd.Series([misspelled, p, r, f1])


# MAIN EXECUTION

if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # 1. TRAIN ON BIG DATA
    processed_data = load_and_label_data(TRAIN_FILE, tokenizer)
    train_data, val_data = train_test_split(processed_data, test_size=0.1, random_state=42)

    train_loader = DataLoader(NERDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NERDataset(val_data), batch_size=BATCH_SIZE)

    best_acc = 0.0
    best_model = None

    for layers in [0, 1, 2, 4]:
        model, acc = train_experiment(layers, train_loader, val_loader)
        if acc > best_acc:
            best_acc = acc
            best_model = model

    print(f"\nBest Validation Accuracy: {best_acc:.4f}")

    # 2. TEST ON SMALL DATA
    print(f"\nEvaluating on Test File: {TEST_FILE}")
    df_test = pd.read_csv(TEST_FILE)
    df_test['text'] = df_test['Title'].fillna('') + " " + df_test['Description'].fillna('')

    if 'Species' in df_test.columns:
        df_test['Ground_Truth_List'] = df_test['Species'].apply(lambda x: [s.strip() for s in str(x).split(',')] if str(x).lower() != 'nan' else [])
    else:
        df_test['Ground_Truth_List'] = [[] for _ in range(len(df_test))]

    best_model.eval()
    tqdm.pandas(desc="Extracting")
    df_test['Extracted_List'] = df_test['text'].progress_apply(lambda x: extract_species(x, best_model, tokenizer))

    metrics = df_test.apply(calculate_row_metrics, axis=1)
    metrics.columns = ['Misspelled Names', 'Precision', 'Recall', 'F1']
    df_test = pd.concat([df_test, metrics], axis=1)

    # 3. PRINT RESULTS
    print("\n" + "="*40)
    print("FINAL PERFORMANCE ON SMALL DATASET")
    print("="*40)
    print(f"Macro Precision: {df_test['Precision'].mean():.4f}")
    print(f"Macro Recall:    {df_test['Recall'].mean():.4f}")
    print(f"Macro F1 Score:  {df_test['F1'].mean():.4f}")

    total_tp, total_fp, total_fn = 0, 0, 0
    for i, row in df_test.iterrows():
        y_true = set(row['Ground_Truth_List'])
        y_pred = set(row['Extracted_List'])
        tp = len(y_true.intersection(y_pred))
        total_tp += tp
        total_fp += len(y_pred - y_true)
        total_fn += len(y_true - y_pred)

    global_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    global_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    global_f1 = 2 * (global_p * global_r) / (global_p + global_r) if (global_p + global_r) > 0 else 0

    print("-" * 30)
    print(f"Total Species in Test Set: {total_tp + total_fn}")
    print(f"Total Correctly Found:     {total_tp}")
    print(f"Total Missed:              {total_fn}")
    print("-" * 30)
    print(f"Global Precision: {global_p:.4f}")
    print(f"Global Recall:    {global_r:.4f}")
    print(f"Global F1 Score:  {global_f1:.4f}")
    print("="*40)

    df_test['Extracted'] = df_test['Extracted_List'].apply(lambda x: ", ".join(x))
    df_test['Ground Truth'] = df_test['Ground_Truth_List'].apply(lambda x: ", ".join(x))
    df_test['Misspelled Names'] = df_test['Misspelled Names'].apply(lambda x: ", ".join(x))

    save_cols = [c for c in df_test.columns if c not in ['Ground_Truth_List', 'Extracted_List']]
    df_test[save_cols].to_csv(OUTPUT_FILE, index=False)
    print(f"Results saved to {OUTPUT_FILE}")

Loading data from species_525_cleaned.csv (412 rows)...
Training: Unfreezing Last 0 Layers


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: NoYo25/BiodivBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Epoch 1:   0%|          | 0/23 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/23 [00:00<?, ?it/s]

...

Epoch 3:   0%|          | 0/23 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/23 [00:00<?, ?it/s]


Best Validation Accuracy: 0.9692

Evaluating on Test File: Data_cleaned.csv


Extracting:   0%|          | 0/44 [00:00<?, ?it/s]


FINAL PERFORMANCE ON SMALL DATASET
Macro Precision: 0.7643
Macro Recall:    0.7413
Macro F1 Score:  0.7466
------------------------------
Total Species in Test Set: 63
Total Correctly Found:     45
Total Missed:              18
------------------------------
Global Precision: 0.5921
Global Recall:    0.7143
Global F1 Score:  0.6475
Results saved to biodivbert_result.csv
